In [7]:
from sentence_transformers import SentenceTransformer
import pandas as pd

In [6]:
model = SentenceTransformer("intfloat/e5-small-v2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6380.22it/s]


In [8]:
df = pd.read_json("./ingest/chunked_data.jsonl", orient="records", lines=True)
df

,uuid,title,doc_id,chunk_index_in_document,text
0,2322a7ba-1b14-4bbb-8080-5bb7776667af,Man About Town,https://en.wikipedia.org/wiki/Man_About_Town_(...,0,"Top Hollywood talent agent, Jack Giamoro (Ben ..."
1,16b41447-6996-436f-9606-91ab3aa8b850,Man of the Year,https://en.wikipedia.org/wiki/Man_of_the_Year_...,0,"Tom Dobbs is host of a satirical news program,..."
2,e2fd6cbe-e717-4c69-a6a4-6a989f5aff19,Man of the Year,https://en.wikipedia.org/wiki/Man_of_the_Year_...,1,and given an injection. The next day she displ...
3,1a203d4d-ac1f-44ce-a23f-0c4340209621,Man of the Year,https://en.wikipedia.org/wiki/Man_of_the_Year_...,2,feeling that is soon justified when she sees D...
4,446f8747-0a69-41b6-9d6c-3540a759bd83,The Marine,https://en.wikipedia.org/wiki/The_Marine,0,"In Iraq, John Triton (John Cena) of the United..."
...,...,...,...,...,...
21887,956f43a5-4044-470a-aa7b-986bc21ea535,Hammett,https://en.wikipedia.org/wiki/Hammett_(film),0,"San Francisco-based Dashiell Hammett, trying t..."
21888,6f6660ae-d54b-4ee4-a4bc-f0387f502326,Hanky Panky,https://en.wikipedia.org/wiki/Hanky_Panky_(198...,0,An architect named Michael Jordon accidentally...
21889,e1e657ca-3f03-4b14-96ce-ce9781767797,Harry Tracy,"https://en.wikipedia.org/wiki/Harry_Tracy,_Des...",0,"By the end of the 19th century, Butch Cassidy,..."
21890,256a0381-f439-46cb-a425-1978a16240e0,A Midsummer Night's Sex Comedy,https://en.wikipedia.org/wiki/A_Midsummer_Nigh...,0,It is the early 1900s. Distinguished philosoph...


In [9]:
BATCH_SIZE = 128

passages = ["passage: " + text for text in df['text'].tolist()]

embeddings = model.encode(
    passages,
    batch_size=BATCH_SIZE,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
)

df['embedding'] = list(embeddings)

Batches: 100%|██████████| 172/172 [05:32<00:00,  1.94s/it]


In [10]:
df

,uuid,title,doc_id,chunk_index_in_document,text,embedding
0,2322a7ba-1b14-4bbb-8080-5bb7776667af,Man About Town,https://en.wikipedia.org/wiki/Man_About_Town_(...,0,"Top Hollywood talent agent, Jack Giamoro (Ben ...","[-0.10754495, 0.029643733, -0.03145323, -0.028..."
1,16b41447-6996-436f-9606-91ab3aa8b850,Man of the Year,https://en.wikipedia.org/wiki/Man_of_the_Year_...,0,"Tom Dobbs is host of a satirical news program,...","[-0.08427628, 0.07323218, 0.00399733, -0.01053..."
2,e2fd6cbe-e717-4c69-a6a4-6a989f5aff19,Man of the Year,https://en.wikipedia.org/wiki/Man_of_the_Year_...,1,and given an injection. The next day she displ...,"[-0.06906479, 0.071653366, -0.0051075784, 0.01..."
3,1a203d4d-ac1f-44ce-a23f-0c4340209621,Man of the Year,https://en.wikipedia.org/wiki/Man_of_the_Year_...,2,feeling that is soon justified when she sees D...,"[-0.078077994, 0.076336816, 0.021279372, 0.001..."
4,446f8747-0a69-41b6-9d6c-3540a759bd83,The Marine,https://en.wikipedia.org/wiki/The_Marine,0,"In Iraq, John Triton (John Cena) of the United...","[-0.07341678, 0.09240317, 0.021616297, 0.01412..."
...,...,...,...,...,...,...
21887,956f43a5-4044-470a-aa7b-986bc21ea535,Hammett,https://en.wikipedia.org/wiki/Hammett_(film),0,"San Francisco-based Dashiell Hammett, trying t...","[-0.06445425, 0.074678235, -0.011070733, 0.002..."
21888,6f6660ae-d54b-4ee4-a4bc-f0387f502326,Hanky Panky,https://en.wikipedia.org/wiki/Hanky_Panky_(198...,0,An architect named Michael Jordon accidentally...,"[-0.06350624, 0.04022677, 0.006863496, 0.02499..."
21889,e1e657ca-3f03-4b14-96ce-ce9781767797,Harry Tracy,"https://en.wikipedia.org/wiki/Harry_Tracy,_Des...",0,"By the end of the 19th century, Butch Cassidy,...","[-0.054992337, 0.03218917, 0.026356624, -0.014..."
21890,256a0381-f439-46cb-a425-1978a16240e0,A Midsummer Night's Sex Comedy,https://en.wikipedia.org/wiki/A_Midsummer_Nigh...,0,It is the early 1900s. Distinguished philosoph...,"[-0.065579735, 0.026074206, 0.0060411408, -0.0..."


In [26]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

client = QdrantClient("localhost", port=6333)

client.recreate_collection(
    collection_name="movies",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

/var/folders/wd/k5bk9g_9623_1td2fb7yf1700000gp/T/ipykernel_30228/695544110.py:6: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [27]:
client.upload_points(
    collection_name="movies",
    points=(
        PointStruct(
            id=row.uuid,
            vector=list(row.embedding),
            payload={
                "uuid": row.uuid,
                "doc_id": row.doc_id,
                "title": row.title,
                "chunk_index": row.chunk_index_in_document,
                "text": row.text,
            },
        )
        for row in df.itertuples()
    ),
    batch_size=256,
    parallel=1,
)

In [22]:
from qdrant_client import QdrantClient



query = """a married woman meets her former lover on the Titanic and has an affair before the ship sinks"""
query_vector = model.encode("query: " + query, normalize_embeddings=True).tolist()

client = QdrantClient("localhost", port=6333)

result = client.query_points(
    collection_name="movies",
    query=query_vector,
    limit=20,
    with_payload=True,
)

for p in result.points:
    print(round(p.score, 3), p.id, p.payload)

0.848 b90c7715-e95a-4658-81d5-4f3491fde761 {'doc_id': 'https://en.wikipedia.org/wiki/Strange_Bedfellows_(1965_film)', 'title': 'Strange Bedfellows', 'chunk_index': 0}
0.836 1e7bb0b7-3455-4188-b6ae-9c193254218b {'doc_id': 'https://en.wikipedia.org/wiki/Out_to_Sea', 'title': 'Out to Sea', 'chunk_index': 1}
0.83 ac632c65-7f18-474a-a883-5a9c032ba4cb {'doc_id': 'https://en.wikipedia.org/wiki/The_Last_Married_Couple_in_America', 'title': 'The Last Married Couple in America', 'chunk_index': 0}
0.827 37d75365-a6fa-4603-9633-616f6663aa12 {'doc_id': 'https://en.wikipedia.org/wiki/Titanic_(1996_TV_miniseries)', 'title': 'Titanic', 'chunk_index': 0}
0.825 11d154bd-e6b0-46c5-97ea-a7b79031c84e {'doc_id': 'https://en.wikipedia.org/wiki/Loving_Couples_(1980_film)', 'title': 'Loving Couples', 'chunk_index': 0}
0.825 58912608-bd6d-4eda-a42d-e45dc164ee68 {'doc_id': 'https://en.wikipedia.org/wiki/The_African_Queen_(film)', 'title': 'The African Queen', 'chunk_index': 3}
0.824 6d39b76c-8c82-4fdf-80ab-01f6b